# Q3 - Player win/lose, K/D distribution and streaks

1. Win/lose and K/D ratio distribution for all players.
2. Max defeat streak and max victory streak (in a row) for players with at least 3 matches.

Choices (from the data quality notebook):
- Win/lose uses only `victory` in `TRUE` / `FALSE`. `draw` / `other` (56 rows) are neither a win nor a loss.
- K/D uses only rows with non-null `kills` and `death` (the 152 rows of the 2025-12-09 incident are left out).
- Nothing else is excluded: all game modes are kept.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

sns.set_theme(style='whitegrid')

DATA_PATH: Path = Path('playersmatches_test - playersmatches_test.csv')
NUM_COLUMNS: list[str] = ['kills', 'death', 'playernb_in_match']
MIN_MATCHES: int = 3

df_player_matches: pl.DataFrame = pl.read_csv(
    DATA_PATH,
    infer_schema=False,
    null_values=['null'],
).with_columns(
    pl.col('match_datetime').str.to_datetime(time_zone='UTC', strict=False),
    pl.col(*NUM_COLUMNS).cast(pl.Int64, strict=False),
    (pl.col('still_active') == 'TRUE').alias('still_active'),
)
print(df_player_matches.shape, '| players:', df_player_matches['playerid'].n_unique())

FileNotFoundError: Le fichier spécifié est introuvable. (os error 2): playersmatches_test - playersmatches_test.csv

## 1. Win / lose per player

In [ ]:
df_player_stats: pl.DataFrame = (
    df_player_matches.group_by('playerid')
    .agg(
        pl.len().alias('matches'),
        (pl.col('victory') == 'TRUE').sum().alias('wins'),
        (pl.col('victory') == 'FALSE').sum().alias('losses'),
        pl.col('kills').sum().alias('kills_sum'),
        pl.col('death').sum().alias('death_sum'),
        pl.col('kills').count().alias('matches_with_kd'),
    )
    .with_columns(
        (pl.col('wins') + pl.col('losses')).alias('decided'),
    )
    .with_columns(
        (pl.col('wins') / pl.col('decided')).alias('win_rate'),
        # K/D = total kills / total deaths over the player's matches (undefined if no death at all)
        pl.when(pl.col('death_sum') > 0).then(pl.col('kills_sum') / pl.col('death_sum')).alias('kd_ratio'),
    )
)
print('players with no decided match (win rate undefined):', df_player_stats.filter(pl.col('decided') == 0).height)
print(df_player_stats.select('matches', 'win_rate', 'kd_ratio').describe())

In [ ]:
# still_active should be a player attribute (constant per player): check, then build player x gamemode stats
print('players with several still_active values:', df_player_matches.group_by('playerid').agg(pl.col('still_active').n_unique().alias('n')).filter(pl.col('n') > 1).height)

GAMEMODE_ORDER: list[str] = ['PVP_RANKED', 'PVP_UNRANKED', 'PVP']  # OPERATIONS (5 rows) left out
MIN_DECIDED_PLOT: int = 10  # below this, a player rate is mostly noise

df_player_mode: pl.DataFrame = (
    df_player_matches.filter(pl.col('gamemode').is_in(GAMEMODE_ORDER))
    .group_by('playerid', 'gamemode', 'still_active')
    .agg(
        pl.len().alias('matches'),
        (pl.col('victory') == 'TRUE').sum().alias('wins'),
        (pl.col('victory') == 'FALSE').sum().alias('losses'),
        pl.col('kills').sum().alias('kills_sum'),
        pl.col('death').sum().alias('death_sum'),
        pl.col('kills').count().alias('matches_with_kd'),
    )
    .with_columns((pl.col('wins') + pl.col('losses')).alias('decided'))
    .with_columns(
        (pl.col('wins') / pl.col('decided')).alias('win_rate'),
        pl.when(pl.col('death_sum') > 0).then(pl.col('kills_sum') / pl.col('death_sum')).alias('kd_ratio'),
        pl.when(pl.col('still_active')).then(pl.lit('Active')).otherwise(pl.lit('Inactive')).alias('activity'),
    )
)
print(df_player_mode.group_by('gamemode', 'activity').len().sort('gamemode', 'activity'))

In [ ]:
# matches played per player: very skewed (1 to ~2000), so log scale
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.violinplot(data=df_player_stats.to_pandas(), y='matches', inner='quartile', cut=0, log_scale=True, color='steelblue', ax=axes[0])
axes[0].set_title(f'Matches played per player - all players (n={df_player_stats.height})')

sns.violinplot(
    data=df_player_mode.to_pandas(), x='gamemode', y='matches', hue='activity', order=GAMEMODE_ORDER, hue_order=['Active', 'Inactive'],
    split=True, inner='quartile', cut=0, density_norm='width', gap=0.05, log_scale=True, ax=axes[1],
)
axes[1].set_title('Matches played per player and game mode, active vs inactive')
axes[1].set_xlabel('')
fig.tight_layout()
plt.show()

print(df_player_stats['matches'].describe())
df_player_mode.group_by('gamemode', 'activity').agg(
    pl.len().alias('players'),
    pl.col('matches').median().alias('matches_median'),
    pl.col('matches').mean().round(1).alias('matches_mean'),
    pl.col('matches').max().alias('matches_max'),
).sort('gamemode', 'activity')

Observation:
- Median of 28 matches per player, but very uneven (max 1,974); a quarter of the players have 5 matches or fewer.
- Inactive players have far fewer matches than active ones in every game mode (e.g. median 26 vs 68.5 in `PVP_RANKED`).

In [ ]:
df_winrate: pl.DataFrame = df_player_stats.drop_nulls('win_rate')
df_winrate_10: pl.DataFrame = df_winrate.filter(pl.col('decided') >= 10)

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
for ax, df_plot, title in [
    (axes[0], df_winrate, f'All players (n={df_winrate.height})'),
    (axes[1], df_winrate_10, f'10+ decided matches (n={df_winrate_10.height})'),
]:
    sns.violinplot(data=df_plot.to_pandas(), y='win_rate', inner='quartile', cut=0, color='steelblue', ax=ax)
    ax.axhline(0.5, color='red', linestyle='--')
    ax.set_title(title)
fig.suptitle('Win rate per player (red = 50 %)')
fig.tight_layout()
plt.show()

print(df_winrate_10['win_rate'].describe())

In [ ]:
# win rate per game mode, split by active / inactive (a player appears once per mode played)
df_plot = df_player_mode.filter(pl.col('decided') >= MIN_DECIDED_PLOT).to_pandas()
fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(
    data=df_plot, x='gamemode', y='win_rate', hue='activity', order=GAMEMODE_ORDER, hue_order=['Active', 'Inactive'],
    split=True, inner='quartile', cut=0, density_norm='width', gap=0.05, ax=ax,
)
ax.axhline(0.5, color='red', linestyle='--')
ax.set_title(f'Win rate per game mode, active vs inactive ({MIN_DECIDED_PLOT}+ decided matches in the mode)')
ax.set_xlabel('')
plt.show()

df_player_mode.filter(pl.col('decided') >= MIN_DECIDED_PLOT).group_by('gamemode', 'activity').agg(
    pl.len().alias('players'),
    pl.col('win_rate').median().round(3).alias('win_rate_median'),
    pl.col('win_rate').mean().round(3).alias('win_rate_mean'),
    pl.col('win_rate').std().round(3).alias('win_rate_std'),
).sort('gamemode', 'activity')

Observation:
- Median win rate of 49%: the typical player wins about half of his matches.
- The wide spread over all players is small-sample noise; with 10+ decided matches it tightens around 50% (5-95% range: 31-65%).
- Median win rate per game mode: `PVP_RANKED` 50%, `PVP_UNRANKED` 53%, `PVP` 45%.
- Identical win rate distribution between active and inactive players, in every game mode.

## 2. K/D ratio per player

In [ ]:
df_kd: pl.DataFrame = df_player_stats.drop_nulls('kd_ratio')
print('players without K/D (no death at all or no kills/death recorded):', df_player_stats.height - df_kd.height)
print(df_kd['kd_ratio'].describe())
print('share of players with K/D >= 1:', round((df_kd['kd_ratio'] >= 1).mean(), 3))

df_kd_10: pl.DataFrame = df_kd.filter(pl.col('matches_with_kd') >= 10)
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
for ax, df_plot, title in [
    (axes[0], df_kd, f'All players (n={df_kd.height})'),
    (axes[1], df_kd_10, f'10+ matches (n={df_kd_10.height})'),
]:
    sns.violinplot(data=df_plot.to_pandas(), y='kd_ratio', inner='quartile', cut=0, color='steelblue', ax=ax)
    ax.axhline(1, color='red', linestyle='--')
    ax.set_title(title)
fig.suptitle('K/D ratio per player (red = 1)')
fig.tight_layout()
plt.show()

In [ ]:
# K/D per game mode, split by active / inactive
df_plot = df_player_mode.drop_nulls('kd_ratio').filter(pl.col('matches_with_kd') >= MIN_DECIDED_PLOT).to_pandas()
fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(
    data=df_plot, x='gamemode', y='kd_ratio', hue='activity', order=GAMEMODE_ORDER, hue_order=['Active', 'Inactive'],
    split=True, inner='quartile', cut=0, density_norm='width', gap=0.05, ax=ax,
)
ax.axhline(1, color='red', linestyle='--')
ax.set_title(f'K/D per game mode, active vs inactive ({MIN_DECIDED_PLOT}+ matches in the mode)')
ax.set_xlabel('')
plt.show()

df_player_mode.drop_nulls('kd_ratio').filter(pl.col('matches_with_kd') >= MIN_DECIDED_PLOT).group_by('gamemode', 'activity').agg(
    pl.len().alias('players'),
    pl.col('kd_ratio').median().round(3).alias('kd_median'),
    pl.col('kd_ratio').mean().round(3).alias('kd_mean'),
    pl.col('kd_ratio').std().round(3).alias('kd_std'),
).sort('gamemode', 'activity')

In [ ]:
# win rate vs K/D, players with 10+ matches
df_scatter = df_kd.filter(pl.col('decided') >= 10, pl.col('matches_with_kd') >= 10).to_pandas()
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=df_scatter, x='kd_ratio', y='win_rate', size='matches', sizes=(10, 150), alpha=0.5, ax=ax)
ax.set_title('Win rate vs K/D per player (10+ matches)')
plt.show()
print('correlation win rate / K/D:', round(df_scatter['win_rate'].corr(df_scatter['kd_ratio']), 3))

Observation:
- Median K/D of 0.86: the median player has more deaths than kills (only 37.7% of players are at 1 or above).
- Identical K/D distribution between active and inactive players, in every game mode.
- K/D is linked to win rate (correlation 0.48 among players with 10+ matches), as expected.

## 3. Max defeat and victory in a row (players with at least 3 matches)

Matches are ordered by `match_datetime` (then `matchid` as tie-break). A `draw` / `other` result breaks a streak.
Streak = run of consecutive identical `victory` values; we keep the longest `TRUE` run (victory) and the longest `FALSE` run (defeat).

In [ ]:
df_eligible: pl.DataFrame = df_player_stats.filter(pl.col('matches') >= MIN_MATCHES).select('playerid', 'matches')
print('eligible players (>= 3 matches):', df_eligible.height, 'of', df_player_stats.height)

df_runs: pl.DataFrame = (
    df_player_matches.join(df_eligible.select('playerid'), on='playerid')
    .sort('playerid', 'match_datetime', 'matchid')
    .with_columns(
        (pl.col('victory') != pl.col('victory').shift(1).over('playerid')).fill_null(True).alias('new_run')
    )
    .with_columns(pl.col('new_run').cum_sum().over('playerid').alias('run_id'))
    .group_by('playerid', 'run_id', 'victory')
    .agg(pl.len().alias('run_length'))
)

df_streaks: pl.DataFrame = (
    df_eligible.join(
        df_runs.filter(pl.col('victory') == 'TRUE').group_by('playerid').agg(pl.col('run_length').max().alias('max_win_streak')),
        on='playerid', how='left',
    )
    .join(
        df_runs.filter(pl.col('victory') == 'FALSE').group_by('playerid').agg(pl.col('run_length').max().alias('max_loss_streak')),
        on='playerid', how='left',
    )
    .with_columns(pl.col('max_win_streak', 'max_loss_streak').fill_null(0))  # never won / never lost
    .sort('matches', descending=True)
)
print(df_streaks.select('matches', 'max_win_streak', 'max_loss_streak').describe())
df_streaks.head(10)

In [ ]:
# sanity check: recompute the streaks with plain python on 50 random players
def python_streaks(results: list[str]) -> tuple[int, int]:
    best = {'TRUE': 0, 'FALSE': 0}
    current, previous = 0, None
    for result in results:
        current = current + 1 if result == previous else 1
        previous = result
        if result in best:
            best[result] = max(best[result], current)
    return best['TRUE'], best['FALSE']


rng = np.random.default_rng(0)
sample_ids = rng.choice(df_streaks['playerid'].to_numpy(), size=50, replace=False)
mismatches = 0
for player_id in sample_ids:
    results = df_player_matches.filter(pl.col('playerid') == player_id).sort('match_datetime', 'matchid')['victory'].to_list()
    expected = python_streaks(results)
    got = df_streaks.filter(pl.col('playerid') == player_id).select('max_win_streak', 'max_loss_streak').row(0)
    mismatches += expected != got
print('mismatches on 50 random players:', mismatches)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, column, color, title in [
    (axes[0], 'max_win_streak', 'seagreen', 'Max victories in a row'),
    (axes[1], 'max_loss_streak', 'indianred', 'Max defeats in a row'),
]:
    sns.countplot(data=df_streaks.to_pandas(), x=column, color=color, ax=ax)
    ax.set_title(f'{title} (n={df_streaks.height} players)')
    ax.set_yscale('log')
fig.tight_layout()
plt.show()

# streaks grow with the number of matches: compare by activity level
(
    df_streaks.with_columns(
        pl.col('matches').cut([9, 29, 99], labels=['3-9', '10-29', '30-99', '100+']).alias('matches_bucket')
    )
    .group_by('matches_bucket')
    .agg(
        pl.len().alias('players'),
        pl.col('max_win_streak').mean().round(2).alias('win_streak_mean'),
        pl.col('max_win_streak').max().alias('win_streak_max'),
        pl.col('max_loss_streak').mean().round(2).alias('loss_streak_mean'),
        pl.col('max_loss_streak').max().alias('loss_streak_max'),
    )
    .sort('players', descending=True)
)

In [ ]:
print('top win streaks')
print(df_streaks.sort('max_win_streak', descending=True).head(5))
print('top loss streaks')
print(df_streaks.sort('max_loss_streak', descending=True).head(5))

Observation:
- Median best run of 4 victories and 5 defeats in a row (700 players with at least 3 matches).
- The more matches a player has, the longer his streaks (mean best win streak: 2.2 for 3-9 matches, 7.4 for 100+).
- Outliers: a win streak of 29 (next highest is 14) and a loss streak of 20, worth a check.
- A streak of 0 means the player never won (or never lost).